# CS-4063 — Natural Language Processing | Assignment 3
## Transformer-based Review Understanding with RAG-Enhanced Explanation Generation
**University:** FAST National University of Computer & Emerging Sciences  
**Course:** CS-4063 Natural Language Processing  



### System Overview
```
Amazon Reviews (3 categories, 30k–45k samples)
          │
          ▼
    Preprocessing  (clean → tokenise → vocab → encode → split)
          │
          ▼
  Part A: Encoder-Only Transformer  ──► embeddings saved to disk
          │                                      │
          ▼                                      ▼
  Part B: Retrieval Module  ◄── cosine similarity on saved embeddings
          │
          ▼
  Part C: Decoder-Only Transformer  (RAG input → explanation text)
```

---
## Section 0 — Environment Setup & Global Hyperparameters

All hyperparameters live in one place so they are easy to tune.  
See **Section 8** for the full hyperparameter tuning log.

In [1]:
import os, re, json, gzip, math, random, time, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from collections import Counter
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
import seaborn as sns
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import OneCycleLR

# ── Reproducibility ────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"{'='*55}")
print(f"  Device        : {DEVICE}")
if torch.cuda.is_available():
    print(f"  GPU Name      : {torch.cuda.get_device_name(0)}")
    print(f"  GPU Memory    : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
print(f"{'='*55}")

os.makedirs('models',  exist_ok=True)
os.makedirs('results', exist_ok=True)
print("\n  models/ and results/ directories ready.")

# ── Hyperparameters ────────────────────────────────────────────────────────
# Encoder
MAX_SEQ_LEN    = 96     # max tokens per review (shorter = much faster)
VOCAB_MIN_FREQ = 4      # prune rare tokens
EMBED_DIM      = 128    # d_model for both encoder and decoder
NUM_HEADS      = 4      # attention heads  (d_k = 32 per head)
NUM_ENC_LAYERS = 2      # encoder depth (2 is fast, 3 gives marginal gains)
FF_DIM         = 256    # FFN hidden size
DROPOUT        = 0.1

# Decoder
DEC_MAX_LEN    = 80     # decoder sequence length
NUM_DEC_LAYERS = 2

# Training
BATCH_SIZE     = 128    # larger batch = faster (use 64 if OOM)
ENC_EPOCHS     = 8      # encoder epochs
DEC_EPOCHS     = 8      # decoder epochs
LR             = 5e-4

# RAG
TOP_K          = 3      # neighbours to retrieve
MAX_GEN_LEN    = 28     # max new tokens at inference

print("\n  Hyperparameters loaded:")
print(f"    MAX_SEQ_LEN={MAX_SEQ_LEN}  EMBED_DIM={EMBED_DIM}  HEADS={NUM_HEADS}")
print(f"    ENC_LAYERS={NUM_ENC_LAYERS}  DEC_LAYERS={NUM_DEC_LAYERS}  FF_DIM={FF_DIM}")
print(f"    BATCH={BATCH_SIZE}  LR={LR}  ENC_EPOCHS={ENC_EPOCHS}  DEC_EPOCHS={DEC_EPOCHS}")
print(f"    TOP_K={TOP_K}  MAX_GEN_LEN={MAX_GEN_LEN}")

  Device        : cpu

  models/ and results/ directories ready.

  Hyperparameters loaded:
    MAX_SEQ_LEN=96  EMBED_DIM=128  HEADS=4
    ENC_LAYERS=2  DEC_LAYERS=2  FF_DIM=256
    BATCH=128  LR=0.0005  ENC_EPOCHS=8  DEC_EPOCHS=8
    TOP_K=3  MAX_GEN_LEN=28


---
## Section 1 — Dataset Loading

We combine three Amazon product categories to ensure domain diversity:

| Category | Domain | Reviews |
|---|---|---|
| **Appliances** | Household appliances (vacuums, refrigerators, etc.) | ≤ 15 000 |
| **Luxury Beauty** | High-end cosmetics, skincare, fragrances | ≤ 15 000 |
| **Video Games** | Gaming consoles, games, peripherals | ≤ 15 000 |

**Total target: 30 000–45 000 samples** as required by the assignment.

Each sample contains:
- `text` — the review body (string)
- `rating` — star rating 1–5 (integer)
- `category` — which product domain (string)

Reviews with fewer than 5 words are discarded as they carry no useful signal.

In [2]:
def load_amazon_gz(filepath, max_samples=15000):
    """
    Stream-load a gzipped Amazon review JSON-lines file.
    We parse line by line to avoid loading the entire file into RAM.
    Only reviews with actual text and a rating are kept.
    """
    records = []
    print(f"  Loading {filepath} ...", end=' ', flush=True)
    t0 = time.time()
    with gzip.open(filepath, 'rt', encoding='utf-8') as fh:
        for line in fh:
            if len(records) >= max_samples:
                break
            try:
                obj    = json.loads(line.strip())
                text   = obj.get('reviewText', '').strip()
                rating = obj.get('overall', None)
                if text and rating is not None and len(text.split()) >= 5:
                    records.append({'text': text, 'rating': int(rating)})
            except (json.JSONDecodeError, KeyError):
                continue
    elapsed = time.time() - t0
    print(f"loaded {len(records):,} reviews in {elapsed:.1f}s")
    return records

# Load all three categories
FILES = {
    'Appliances':    'Appliances.json.gz',
    'Luxury_Beauty': 'Luxury_Beauty.json.gz',
    'Video_Games':   'Video_Games.json.gz',
}

print("Loading Amazon Review datasets...")
print("-" * 50)
all_records = []
for category, fname in FILES.items():
    batch = load_amazon_gz(fname, max_samples=15000)
    for r in batch:
        r['category'] = category
    all_records.extend(batch)

# Shuffle to mix categories evenly
random.shuffle(all_records)

print("-" * 50)
print(f"  Total reviews loaded : {len(all_records):,}")
print()

# Per-category breakdown
cat_counts = Counter(r['category'] for r in all_records)
for cat, cnt in cat_counts.items():
    print(f"    {cat:20s}: {cnt:,} reviews  ({cnt/len(all_records)*100:.1f}%)")

# Rating distribution
print()
print("  Rating distribution across all categories:")
rc = Counter(r['rating'] for r in all_records)
for star in sorted(rc):
    bar = '█' * (rc[star] // 500)
    print(f"    {star}★  {bar}  {rc[star]:,}")

Loading Amazon Review datasets...
--------------------------------------------------
  Loading Appliances.json.gz ... loaded 15,000 reviews in 0.1s
  Loading Luxury_Beauty.json.gz ... loaded 15,000 reviews in 0.1s
  Loading Video_Games.json.gz ... loaded 15,000 reviews in 0.2s
--------------------------------------------------
  Total reviews loaded : 45,000

    Luxury_Beauty       : 15,000 reviews  (33.3%)
    Video_Games         : 15,000 reviews  (33.3%)
    Appliances          : 15,000 reviews  (33.3%)

  Rating distribution across all categories:
    1★  ███████  3,904
    2★  ███  1,965
    3★  ██████  3,330
    4★  █████████████  6,740
    5★  ██████████████████████████████████████████████████████████  29,061
